In [ ]:
!pip install -q ultralytics

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import shutil
from pathlib import Path

DATASET_DIR = Path('/content/vision_dataset')
raw_images  = DATASET_DIR / 'raw_images'
raw_labels  = DATASET_DIR / 'labels'

dst_images = DATASET_DIR / 'images' / 'train'
dst_labels = DATASET_DIR / 'labels' / 'train'  # ← CORREGIDO: YOLOv8 espeja images/ con labels/

dst_images.mkdir(parents=True, exist_ok=True)
dst_labels.mkdir(parents=True, exist_ok=True)

CLASES_DISPONIBLES = ['armas', 'botellas', 'dinero', 'animales']
EXTENSIONES_VALIDAS = {'.jpg', '.jpeg', '.png', '.bmp'}

for clase in CLASES_DISPONIBLES:
    img_dir = raw_images / clase
    if img_dir.exists():
        count = 0
        for img in img_dir.glob('*'):
            if img.suffix.lower() in EXTENSIONES_VALIDAS:
                shutil.copy2(img, dst_images / img.name)
                count += 1
        print(f'  {clase}: {count} imágenes')
    else:
        print(f'  {clase}: carpeta no encontrada')

    lbl_dir = raw_labels / clase
    if lbl_dir.exists():
        for lbl in lbl_dir.glob('*.txt'):
            shutil.copy2(lbl, dst_labels / lbl.name)

imgs = list(dst_images.glob('*'))
lbls = list(dst_labels.glob('*.txt'))
print(f'\nTotal imágenes: {len(imgs)}')
print(f'Total labels  : {len(lbls)}')

In [ ]:
import yaml
from pathlib import Path

DATASET_DIR = Path('/content/vision_dataset')

NOMBRES_CLASES = [
    'armas',     # 0 — TIER 1
    'botellas',  # 1 — TIER 2
    'dinero',    # 2 — TIER 2
    'animales',  # 3 — TIER 2
]

config = {
    'path'  : str(DATASET_DIR),
    'train' : 'images/train',
    'val'   : 'images/train',   # ← mismo split por ahora, luego separa
    'nc'    : len(NOMBRES_CLASES),
    'names' : NOMBRES_CLASES,
}

YAML_PATH = DATASET_DIR / 'dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print('YAML generado:')
!cat /content/vision_dataset/dataset.yaml

In [ ]:
import yaml
from pathlib import Path

DATASET_DIR = Path('/content/vision_dataset')

# Con lo que tienes ahora:
# Tier 1: armas (idx 0)
# Tier 2: botellas (idx 1), dinero (idx 2), animales (idx 3)
# Tier 3: vacío por ahora — se agrega cuando tengas datos

NOMBRES_CLASES = [
    'armas',     # 0 — TIER 1
    'botellas',  # 1 — TIER 2
    'dinero',    # 2 — TIER 2
    'animales',  # 3 — TIER 2
]

config = {
    'path'  : str(DATASET_DIR),
    'train' : 'images/train',
    'val'   : 'images/train',
    'nc'    : len(NOMBRES_CLASES),
    'names' : NOMBRES_CLASES,
}

YAML_PATH = DATASET_DIR / 'dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print('YAML generado:')
!cat /content/vision_dataset/dataset.yaml

In [ ]:
from pathlib import Path

dst_images = Path('/content/vision_dataset/images/train')
dst_labels = Path('/content/vision_dataset/labels_train')

imgs = {f.stem for f in dst_images.glob('*')}
lbls = {f.stem for f in dst_labels.glob('*.txt')}

print(f'Imágenes con label : {len(imgs & lbls)}')
print(f'Imágenes sin label : {len(imgs - lbls)}')
print(f'Labels sin imagen  : {len(lbls - imgs)}')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

model.train(
    data     = '/content/vision_dataset/dataset.yaml',
    imgsz    = 640,
    epochs   = 50,
    batch    = 16,
    patience = 20,
    device   = 'cuda',
    project  = '/content/runs',
    name     = 'vision_tactical_v1',
    exist_ok = True,
    save     = True,
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    degrees  = 10.0,
    flipud   = 0.0,
    fliplr   = 0.5,
    mosaic   = 1.0,
)

print('Entrenamiento completo.')

In [ ]:
# Índices según el YAML actual
TIER1_IDX = {0}        # armas
TIER2_IDX = {1, 2, 3}  # botellas, dinero, animales
TIER3_IDX = set()      # vacío hasta tener vehiculos, grupos, entornos

NOMBRES = ['armas', 'botellas', 'dinero', 'animales']

def evaluar_alerta(resultados):
    tier1_det = set()
    tier2_det = set()
    tier3_det = set()

    for r in resultados:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            if conf < 0.40:
                continue
            nombre = NOMBRES[cls_id]
            if cls_id in TIER1_IDX:
                tier1_det.add(nombre)
            elif cls_id in TIER2_IDX:
                tier2_det.add(nombre)

    t1 = len(tier1_det)
    t2 = len(tier2_det)

    if t1 >= 1 and t2 >= 2:
        nivel = '🔴 ALERTA RECLUTAMIENTO'
    elif t1 >= 1 and t2 >= 1:
        nivel = '🟠 FLAG ASPIRACIONALIDAD'
    elif t1 >= 1:
        nivel = '🟡 FLAG PASIVO'
    else:
        nivel = '🟢 SIN ALERTA'

    return {
        'nivel' : nivel,
        'tier1' : list(tier1_det),
        'tier2' : list(tier2_det),
    }

print('Motor de scoring listo.')
print()
print('Clases actuales:')
print('  Tier 1: armas')
print('  Tier 2: botellas, dinero, animales')
print('  Tier 3: pendiente (vehiculos, uniformes, grupos, entornos...)')

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/vision_tactical_v1/weights/best.pt')
model.export(format='tflite', imgsz=640)

!find /content/runs/vision_tactical_v1/weights -name '*.tflite'

In [ ]:
from google.colab import files
from pathlib import Path

output_zip = '/content/modelo_tactical_v1.zip'
!zip -j {output_zip} /content/runs/vision_tactical_v1/weights/best.pt

for f in Path('/content/runs/vision_tactical_v1/weights').glob('*.tflite'):
    !zip -j {output_zip} {f}

files.download(output_zip)